In [1]:
# first prompt for getting values
# second prompt for generating of code

In [2]:
import sys
sys.path.append("../src")
import pandas as pd
from utils.utils import generate, eval_generic, to_csv
from utils.models import get_models
import json
import subprocess


In [3]:
models = get_models()
df = pd.read_excel("../data/ai_lab_1.xlsx")
gr_truth = pd.read_csv("../data/ground_truth.csv")

In [4]:
def make_extract_prompt(pid, text: str) -> str:
    return f"""
You are a deterministic JSON extraction engine.

Return EXACTLY one JSON object and NOTHING else.

Task:
1) Extract all AREA measurements only.
   Supported units:
    - square meters (m2, sqm, square meters)
    - square feet (sq ft, sqft, ft2)
    - hectares (ha)
    - acres (ac)
2) IGNORE PERIMETERS, DISTANCES, HEIGHTS, DIAMETERS.
3) Output ONLY VALID JSON array and NOTHING else.
4) NOT generate code

Output format:
[{{"property_id": number, "value": number, "unit": string}}]

Examples:
Input: <pid>12</pid> <text>"Plot is 35000 sq ft, perimeter 500 meters, garden 200 m2"</text>
Valid Output: [{{"property_id": 12, "value": 35000, "unit": "sqft"}}, {{"property_id": 12, "value": 200, "unit": "m2"}}]

Input: <pid>29</pid> <text>"2 acres of land, 500 meters from school, small yard of 50 sqm"</text>
Valid Output: [{{"property_id": 29, "value": 2, "unit": "ac"}}, {{"property_id": 29, "value": 50, "unit": "sqm"}}]

Input: <pid>34</pid> <text>"Plot is 35000 sq ft, perimeter 500 meters, garden 200 m2"</text>
Invalid Output: [{{"property_id": 34, "value": 35000, "unit": "sqft"}}, {{"property_id": 34, "value": 500, "unit": "m"}}, {{"property_id": 34, "value": 200, "unit": "m2"}}]

Input: <pid>56</pid> <text>"Plot is 4000 sq ft, perimeter 2,100 meters"</text>
Invalid Output: [{{"property_id": 56, "value": 4000, "unit": "sqft"}}, {{"property_id": 56, "value": 2100, "unit": "m"}}]

Input: <pid>87</pid> <text>"Plot is 4000 sq ft, perimeter 600 meters"</text>
Invalid Output: [{{"property_id": 87, "value": 4000, "unit": "sqft"}}, {{"property_id": 87, "value": 600, "unit": "m"}}]

Input: <pid>7</pid> <text>"Plot is 4000 sq ft, perimeter 2,100 meters"</text>
Invalid Output: [{{"value": 4000, "unit": "sqft"}}]


NOT generate code!
<pid>
{pid}
</pid>
<text>
{text}
</text>
""".strip()

In [5]:
def make_code_gen_prompt() -> str:
    return """
You are a Python code generator.

`import json` and the variable `data` are already defined before your code runs.
Do NOT redefine them. Do NOT add import statements.

`data` is a list of dicts like:
[{"property_id": 2, "value": 45000, "unit": "sqft"}]

Task: convert each area value to square meters, group by property_id, sum, then print result as JSON.

Conversions:
- m2, sqm, square meters -> no conversion needed
- sq ft, sqft, ft2       -> multiply by 0.092903
- ha                     -> multiply by 10000
- ac                     -> multiply by 4046.86

Output: last line must be exactly:
print(json.dumps(result))
where result = {str(property_id): total_m2}

Example final output: {"2": 4180.635}

Return ONLY the logic code. No imports. No markdown. No explanation.
""".strip()


In [6]:
def make_preamble(json_data: list[dict]) -> str:
    """Deterministic header injected before model code — not trusted to the model."""
    json_str = json.dumps(json_data)
    # repr() safely handles any special chars in the JSON string
    return f"import json\ndata = json.loads({repr(json_str)})\n"

In [7]:
ALLOWED_UNITS = {"m2", "sqm", "square meters", "sqft", "sq ft", "ft2", "ha", "ac"}

def validate(raw_json: str):
    """Parse and structurally validate the extracted JSON."""
    try:
        data = json.loads(raw_json)
    except json.JSONDecodeError as e:
        print("JSON parse error:", e)
        return None, "json_parse_error"

    if not isinstance(data, list) or len(data) == 0:
        print("Validation error: expected non-empty list, got:", type(data))
        return None, "invalid_structure"

    required_keys = {"property_id", "value", "unit"}
    filtered = []
    for i, item in enumerate(data):
        if not isinstance(item, dict):
            print(f"Validation error: item {i} is not a dict, skipping")
            continue
        missing = required_keys - item.keys()
        if missing:
            print(f"Validation error: item {i} missing keys {missing}, skipping")
            continue
        if item["unit"].lower() not in ALLOWED_UNITS:
            print(f"Skipping unsupported unit: {item['unit']!r} in item {i}")
            continue
        filtered.append(item)

    if not filtered:
        print("Validation error: no valid area items after filtering")
        return None, "no_valid_items"

    return filtered, None

In [8]:
def extract_code(text: str) -> str:
    text = text.strip()
    if "```" in text:
        lines = text.split("\n")
        code_lines = []
        inside_block = False
        for line in lines:
            if line.strip().startswith("```"):
                inside_block = not inside_block
                continue
            if inside_block:
                code_lines.append(line)
        return "\n".join(code_lines).strip()
    return text


In [9]:
def exec_code(generated_code: str, preamble: str):
    logic = extract_code(generated_code)
    full_code = preamble + "\n" + logic
    try:
        result = subprocess.run(
            ["python", "-c", full_code],
            capture_output=True,
            text=True,
            timeout=10
        )
        return result
    except Exception as e:
        print("Execution error:", e)
        return None

In [10]:
def parse_stdout(stdout: str) -> dict | None:
    try:
        return json.loads(stdout.strip())
    except json.JSONDecodeError as e:
        print("stdout parse error:", e, "| raw:", repr(stdout))
        return None

In [11]:
def generate_pal(model, pid, prompt):
    # Step 1: extract area measurements
    extract_prompt = make_extract_prompt(pid, prompt)
    raw_json = generate(model, extract_prompt)
    json_data, err = validate(raw_json)

    if err is not None:
        print(f"Extraction failed ({err}), skipping pid={pid}")
        return None

    print("Extracted:", json_data)

    # Step 2: generate conversion logic (no data in prompt)
    code_prompt = make_code_gen_prompt()
    generated_code = generate(model, code_prompt)

    # Step 3: inject preamble + execute
    preamble = make_preamble(json_data)
    res = exec_code(generated_code, preamble)

    if res is None:
        print("Code execution returned None")
        return None

    if res.returncode == 0:
        print(res.stdout)
        return parse_stdout(res.stdout)
    else:
        print("Code execution error:", res.stderr)
        return None


In [12]:
for model in models:
    result = generate_pal(
        model=model.model_name,
        pid=2,
        prompt="In a prestigious residential area, this plot covers 45,000 square feet with a 600-meter perimeter. Two kilometers from the nearest school and shopping center, it's ideal for family homes. At an elevation of 100 meters, the climate is pleasant year-round. The land supports luxury residences or a small community and includes a historic building from the early 1900s, standing 12 meters tall."
    )
    print(f"[{model.short_name}] result:", result)


Skipping unsupported unit: 'm' in item 1
Extracted: [{'property_id': 2, 'value': 45000, 'unit': 'sqft'}]
Code execution error: Traceback (most recent call last):
  File "<string>", line 14, in <module>
NameError: name 'result' is not defined

[llama] result: None
Extracted: [{'property_id': 2, 'value': 45000, 'unit': 'sqft'}]
{"2": 4180.635}

[gemma] result: {'2': 4180.635}
